# Fitxers JSON

FES · UAB · 2026–2027

## Objectius d'aprenentatge

- Representar configuracions i històrics amb JSON.
- Desar amb `dump` i recuperar amb `load`.
- Gestionar errors de sintaxi i reconèixer la necessitat de validar les dades.

Contingut essencial · 30 minuts. La validació detallada de l'esquema és ampliació.

## Dades natives i JSON

JSON representa objectes amb claus textuals, arrays, nombres, text, booleans i `null`. Python els recupera com a diccionaris, llistes, nombres, cadenes, booleans i `None`.

Una tupla es recupera com una llista. Els objectes de classes pròpies necessiten una conversió explícita a dades natives.

In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

configuracio = {"dispositiu": "Estació UAB", "actiu": True, "periode_s": 5}
print(json.dumps(configuracio, ensure_ascii=False, indent=2))


## Escriure i llegir una configuració

`dump(dades, fitxer)` escriu al recurs obert i `load(fitxer)` recupera el valor. `ensure_ascii=False` conserva els accents llegibles i `indent=2` facilita inspeccionar el fitxer.

`dumps` i `loads` treballen amb cadenes. `dump` i `load` treballen amb fitxers.

In [ ]:
with TemporaryDirectory() as carpeta:
    ruta = Path(carpeta) / "configuracio.json"
    with open(ruta, "w", encoding="utf-8") as fitxer:
        json.dump(configuracio, fitxer, ensure_ascii=False, indent=2)
    with open(ruta, "r", encoding="utf-8") as fitxer:
        recuperada = json.load(fitxer)
    print(recuperada)
    print(type(recuperada["actiu"]).__name__)  # bool


## Històric de lectures

Un històric és una llista de diccionaris amb `sensor`, `valor` i `unitat`, igual que al CSV. Això permet reutilitzar el contracte de dades.

`allow_nan=False` rebutja mesures no finites. Preparar el text abans d'obrir el destí detecta errors de conversió abans de substituir el fitxer.

In [ ]:
def desar_json(ruta, lectures):
    """Desa lectures natives amb nombres finits."""
    text = json.dumps(lectures, ensure_ascii=False, indent=2, allow_nan=False)
    with open(ruta, "w", encoding="utf-8") as fitxer:
        fitxer.write(text)


def carregar_json(ruta):
    """Recupera les dades JSON per validar-les després."""
    with open(ruta, "r", encoding="utf-8") as fitxer:
        return json.load(fitxer)

lectures = [{"sensor": "T1", "valor": 21.5, "unitat": "°C"}]
with TemporaryDirectory() as carpeta:
    ruta = Path(carpeta) / "lectures.json"
    desar_json(ruta, lectures)
    print(carregar_json(ruta))
    desar_json(ruta, [])
    print(carregar_json(ruta))  # []


## Sintaxi i estructura

Un document incomplet genera `json.JSONDecodeError`, una subclasse de `ValueError`. Un document amb `{"actiu": true}` és JSON correcte però incompleix el format d'una llista de lectures.

El fitxer de teoria inclou `llegir_json`, amb validació detallada dels camps. Aquest fragment introdueix `isinstance`, que comprova el tipus d'un valor.

In [ ]:
with TemporaryDirectory() as carpeta:
    ruta = Path(carpeta) / "lectures.json"
    ruta.write_text('{"sensor":', encoding="utf-8")
    try:
        carregar_json(ruta)
    except json.JSONDecodeError:
        print("JSON incomplet")

    ruta.write_text('{"actiu": true}', encoding="utf-8")
    dades = carregar_json(ruta)
    if not isinstance(dades, list):
        print("El sistema espera una llista de lectures")


## Cas límit dels nombres no finits

`float("nan")` representa una mesura indefinida. `float("inf")` representa infinit. El JSON estàndard requereix que els registres del nostre sistema continguin nombres finits.

La validació protegeix la persistència de dades que no es podran utilitzar com a mesures.

In [ ]:
with TemporaryDirectory() as carpeta:
    ruta = Path(carpeta) / "lectures.json"
    desar_json(ruta, lectures)
    try:
        desar_json(ruta, [{"sensor": "T1", "valor": float("nan"), "unitat": "°C"}])
    except ValueError:
        print("Mesura rebutjada")
    print(carregar_json(ruta))  # Es recupera l'històric anterior.


## Pràctica

Resol els exercicis 1 i 2 i després el 3 de gestió d'errors. El repte 4 amplia un històric en un únic procés.

[Enunciats](03_json_exercicis.py) · [Solucions](03_json_solucions.py)

Com convertiries l'estat d'un futur objecte `Sensor` a un diccionari que es pugui desar?